# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


# Notebook 4 — Dynamic, frictional linear-pusher identification

**Goal:** identify constant mass and Coulomb friction from noisy object-position histories and known position commands. The object retains velocity between steps.

This notebook is self-contained: it uses NumPy and Matplotlib, runs on CPU, and does not modify or import either original repository. It implements our proposed **implicit-Euler dynamic extension**, not the original `QuasidynamicLinearPusher` class or the adaptive-SLS controller. Keep notebooks 1–3 as baselines.

Run code cells 1–10 in order. Cell 11 optionally exports a new timestamped result folder. Initial results are a **matched discrete-model simulation**, not evidence of physical-robot accuracy. Code cell 10 explicitly checks timestep sensitivity.

Assumptions: horizontal one-dimensional motion; normal load $mg$; known, ideal massless spring actuator; one equal static/kinetic Coulomb coefficient; fixed parameters; exact commanded setpoint; bounded additive position noise. A real actuator, separate static/kinetic friction, and certified model-error bounds are future work.


## Code cell 1 — Environment and known settings


In [ ]:
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt

print("Python executable:", sys.executable)
print("NumPy:", np.__version__)

K_A = 20.0                  # N/m; NEW dynamic experiment, previously 1 N/m
H = 0.01                    # seconds; timestep sensitivity checked in cell 10
W = 0.1                     # m; center separation at contact
GRAVITY = 9.81              # m/s^2
MASS_PRIOR = (0.005, 0.030) # kg
MU_PRIOR = (0.05, 0.60)     # dimensionless
POSITION_NOISE = 5e-6       # m, bound on EACH position measurement
MODEL_ALLOWANCE = 1e-10     # m, roundoff allowance for this matched discrete model ONLY
SEED = 7

assert K_A > 0 and H > 0 and 0 < MASS_PRIOR[0] < MASS_PRIOR[1]
assert 0 <= MU_PRIOR[0] < MU_PRIOR[1]
print("Known stiffness:", K_A, "N/m; timestep:", H, "s")
print("Position noise bound:", POSITION_NOISE * 1e6, "um per sample")


## Code cell 2 — Physical plant

For object state $(o_k,\nu_k)$ and commanded pusher setpoint $u_k$, solve

$$\min_{p^+,o^+}\quad \frac{K_a}{2}(p^+-u_k)^2+
\frac{m}{2h^2}(o^+-o_k-h\nu_k)^2+\mu mg|o^+-o_k|,
\qquad o^+-p^+-w\geq0.$$

Then $\nu^+=(o^+-o_k)/h$ and $\lambda=K_a(u_k-p^+)\geq0$.

Eliminating $p^+$ gives $p^+=\min(u_k,o^+-w)$. The remaining objective is strictly convex and piecewise quadratic in object displacement. Its minimum is either a quadratic stationary point or a kink. The implementation checks that finite candidate set; no optimizer package is required.

Sticking, sliding, and separation are all simulated. The estimator will use only position-certified forward-sliding contact samples.


In [ ]:
def dynamic_step(state, command, mass, mu, h=H, stiffness=K_A, width=W):
    """State = [object position (m), object velocity (m/s)]."""
    o, velocity = np.asarray(state, dtype=float)
    if mass <= 0 or mu < 0 or h <= 0 or stiffness <= 0:
        raise ValueError("Invalid physical parameter.")
    a = mass / h**2
    free_displacement = h * velocity
    contact_offset = float(command) + width - o
    friction = mu * mass * GRAVITY

    # Kinks: zero displacement; command/contact boundary.
    candidates = [0.0, contact_offset]
    for active in (0.0, 1.0):
        for direction in (-1.0, 1.0):
            root = (a * free_displacement + active * stiffness * contact_offset
                    - direction * friction) / (a + active * stiffness)
            if direction * root >= -1e-14:
                mode_ok = root <= contact_offset if active else root >= contact_offset
                if mode_ok:
                    candidates.append(root)

    def energy(displacement):
        compression = max(contact_offset - displacement, 0.0)
        return (0.5 * stiffness * compression**2
                + 0.5 * a * (displacement - free_displacement)**2
                + friction * abs(displacement))

    d = min(candidates, key=energy)
    o_next = o + d
    p_next = min(float(command), o_next - width)
    force = stiffness * max(contact_offset - d, 0.0)
    next_state = np.array([o_next, d / h])
    return next_state, p_next, force


def rollout(commands, mass, mu, initial_state, h=H):
    states = np.empty((len(commands) + 1, 2))
    pushers = np.empty(len(commands))
    forces = np.empty(len(commands))
    states[0] = initial_state
    for k, command in enumerate(commands):
        states[k+1], pushers[k], forces[k] = dynamic_step(
            states[k], command, mass, mu, h=h)
    return {"states": states, "pushers": pushers, "forces": forces}

print("Dynamic plant ready; state is [object position, object velocity].")


## Code cell 3 — Physics checks before identification


In [ ]:
# Free motion with zero friction must preserve incoming velocity.
s, p, force = dynamic_step([0.2, 0.1], -1.0, 0.015, 0.0)
assert np.allclose(s, [0.2 + H * 0.1, 0.1], atol=1e-12)
assert force == 0.0

# Friction during release must slow the object; it must not reverse it.
expected_v = max(0.0, 0.1 - 0.25 * GRAVITY * H)
s, _, force = dynamic_step([0.2, 0.1], -1.0, 0.015, 0.25)
assert np.allclose(s, [0.2 + H * expected_v, expected_v], atol=1e-12)
s, _, _ = dynamic_step([0.2, 0.001], -1.0, 0.015, 0.25)
assert abs(s[1]) < 1e-12

# A spring force below static friction must leave the object at rest.
mass, mu = 0.015, 0.25
small_deflection = 0.5 * mu * mass * GRAVITY / K_A
s, p, force = dynamic_step([0.2, 0.0], 0.2 - W + small_deflection, mass, mu)
assert abs(s[0] - 0.2) < 1e-12 and abs(s[1]) < 1e-12
assert 0 <= force <= mu * mass * GRAVITY

# During forward sliding, contact and momentum balance must hold.
s, p, force = dynamic_step([0.2, 0.0], 0.2 - W + 0.02, mass, mu)
assert s[1] > 0 and force > 0
assert s[0] - p - W >= -1e-12
assert abs(force * (s[0] - p - W)) < 1e-12
assert abs(mass * s[1] / H - (force - mu * mass * GRAVITY)) < 1e-10

print("PASS: inertia, frictional slowing, stopping, sticking, contact, and force balance.")


## Code cell 4 — Continuous experiment and noisy position histories

The command is a prescribed sequence of smooth ramps and holds, followed by release. It is **not yet an optimized nominal trajectory**. Its absolute position is anchored using the first noisy position reading, not hidden true position. The object is never reset between samples.

Only `measured_positions` and `commands` enter identification. True mass, friction, velocity, pusher force, and noiseless positions are used by the simulated plant or evaluation only. Noise is sampled once per position reading, so overlapping three-position windows correctly share measurement errors.


In [ ]:
def smooth_schedule(times, observed_start, knot_times, offsets):
    """Raised-cosine interpolation of absolute position setpoints."""
    knot_times = np.asarray(knot_times, dtype=float)
    offsets = np.asarray(offsets, dtype=float)
    index = np.clip(np.searchsorted(knot_times, times, side="right") - 1,
                    0, len(knot_times) - 2)
    fraction = np.clip((times - knot_times[index]) /
                       (knot_times[index+1] - knot_times[index]), 0, 1)
    blend = 0.5 * (1 - np.cos(np.pi * fraction))
    return observed_start - W + offsets[index] + blend * (offsets[index+1] - offsets[index])


TRUE_MASS = 0.015  # plant and evaluation only
TRUE_MU = 0.25     # plant and evaluation only
INITIAL_STATE = np.array([0.2, 0.0])  # object is initially at rest
KNOT_TIMES = (2 * np.array([0, .15, .40, .60, .90, 1.10, 1.35, 1.55, 1.80, 2.00, 2.25, 2.50])).tolist()
OFFSETS = [-.003, -.003, .018, .018, .045, .045, .070, .070, .095, .095, -.003, -.003]
times = np.arange(round(KNOT_TIMES[-1] / H) + 1) * H
rng = np.random.default_rng(SEED)
position_errors = rng.uniform(-POSITION_NOISE, POSITION_NOISE, len(times))
observed_start = INITIAL_STATE[0] + position_errors[0]
commands = smooth_schedule(times[:-1], observed_start, KNOT_TIMES, OFFSETS)
experiment = rollout(commands, TRUE_MASS, TRUE_MU, INITIAL_STATE)
measured_positions = experiment["states"][:, 0] + position_errors

gap = experiment["states"][1:, 0] - experiment["pushers"] - W
assert np.isfinite(experiment["states"]).all()
assert gap.min() >= -1e-12
assert np.max(abs(gap * experiment["forces"])) < 1e-10
print("Transitions:", len(commands), "  Duration:", times[-1], "s")
print("Actual object travel:", 1000 * (experiment["states"][-1,0] - INITIAL_STATE[0]), "mm")
print("Maximum simulated force:", experiment["forces"].max(), "N")
print("Final simulated speed:", experiment["states"][-1,1], "m/s")


## Code cell 5 — Inspect motion, force, and sensor data


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
axes[0,0].plot(times, 1000 * (experiment["states"][:,0] - INITIAL_STATE[0]), label="True object")
axes[0,0].scatter(times, 1000 * (measured_positions - INITIAL_STATE[0]), s=5, label="Noisy position")
axes[0,0].step(times[:-1], 1000 * (commands + W - INITIAL_STATE[0]), where="post", label="Command + w")
axes[0,0].set_ylabel("Position relative to initial object [mm]")
axes[0,0].legend()
axes[0,1].plot(times, experiment["states"][:,1])
axes[0,1].set_ylabel("True velocity, evaluation only [m/s]")
axes[1,0].plot(times[1:], experiment["forces"], label="Pushing force")
axes[1,0].axhline(TRUE_MU * TRUE_MASS * GRAVITY, color="k", ls="--", label="Sliding friction magnitude")
axes[1,0].set_ylabel("Force, evaluation only [N]")
axes[1,0].legend()
axes[1,1].plot(times, 1e6 * (measured_positions - experiment["states"][:,0]))
axes[1,1].axhline(POSITION_NOISE * 1e6, color="k", ls="--")
axes[1,1].axhline(-POSITION_NOISE * 1e6, color="k", ls="--")
axes[1,1].set_ylabel("Position measurement error [um]")
for ax in axes.flat:
    ax.set_xlabel("Time [s]")
    ax.grid(alpha=.3)
plt.show()


## Code cell 6 — Bounded two-parameter estimator

During forward-sliding contact,
$$o_{k+1}-2o_k+o_{k-1}=\alpha(u_k+w-2o_k+o_{k-1})-g h^2\eta,$$
$$\alpha=\frac{K_a h^2}{m+K_a h^2},\qquad \eta=\mu(1-\alpha).$$
Here $\eta=\gamma/(gh^2)$ is just a rescaled version of our previously proposed friction term $\gamma$. This scaling improves numerical conditioning.

Recover physical parameters using
$$m=K_a h^2(1/\alpha-1),\qquad \mu=\eta/(1-\alpha).$$

The prior is a polygon: $\alpha\in[\alpha_{min},\alpha_{max}]$ and
$\mu_{min}(1-\alpha)\leq\eta\leq\mu_{max}(1-\alpha)$.

For noisy positions $y_i=o_i+n_i$, $|n_i|\leq\epsilon$, define
$$z_k=y_{k+1}-2y_k+y_{k-1},\qquad \zeta_k=u_k+w-2y_k+y_{k-1}.$$
The residual is
$$z_k-\alpha\zeta_k+gh^2\eta
=n_{k+1}-2(1-\alpha)n_k+(1-\alpha)n_{k-1}+\delta_k.$$
Thus a valid bound is $(4-3\alpha)\epsilon+\epsilon_{model}$. Both measured sides are noisy: treating the regressor as exact would be incorrect.

Each accepted sample supplies two **linear half-plane constraints** in $(\alpha,\eta)$. Intersecting them with the current polygon gives a bounded feasible set. This is deterministic set-membership estimation, not a Gaussian confidence interval. It conservatively allows bounded errors for each transition; it does not jointly optimize a single shared sensor-noise sequence over all transitions.

Mode gating also uses only observations: accept a sample only if $y_{k+1}-y_k>2\epsilon$ and $u_k+w-y_{k+1}>\epsilon$, with a small numerical margin. These certify positive displacement and spring compression under this plant model. At least two sufficiently distinct regressor values are needed to separate the parameters.


In [ ]:
def clip_polygon(vertices, normal, bound):
    """Intersect a convex polygon with normal @ theta <= bound."""
    vertices = np.asarray(vertices, dtype=float)
    normal = np.asarray(normal, dtype=float)
    result = []
    for a, b in zip(vertices, np.roll(vertices, -1, axis=0)):
        fa, fb = normal @ a - bound, normal @ b - bound
        a_inside, b_inside = fa <= 0, fb <= 0
        if a_inside:
            result.append(a)
        if a_inside != b_inside:
            result.append(a + (fa / (fa - fb)) * (b - a))
    if not result:
        raise RuntimeError("Empty feasible set: inspect mode, noise bound, and physics. Do not silently widen bounds.")
    return np.asarray(result)


def initial_polygon(mass_prior=MASS_PRIOR, mu_prior=MU_PRIOR):
    lo = K_A * H**2 / (mass_prior[1] + K_A * H**2)
    hi = K_A * H**2 / (mass_prior[0] + K_A * H**2)
    return np.array([[lo, mu_prior[0]*(1-lo)], [hi, mu_prior[0]*(1-hi)],
                     [hi, mu_prior[1]*(1-hi)], [lo, mu_prior[1]*(1-lo)]])


def physical_parameters(theta):
    theta = np.asarray(theta)
    alpha, eta = theta[...,0], theta[...,1]
    return np.stack([K_A * H**2 * (1/alpha - 1), eta / (1-alpha)], axis=-1)


def physical_bounds(polygon):
    # The linear-fractional maps attain extrema at polygon vertices here.
    values = physical_parameters(polygon)
    return np.stack([values.min(axis=0), values.max(axis=0)], axis=1)


def identify(positions, inputs, noise_bound=POSITION_NOISE, allowance=MODEL_ALLOWANCE):
    """Receives NO true mass, friction, velocity, force, or true position."""
    polygon = initial_polygon()
    history = [physical_bounds(polygon)]
    sample_numbers = [0]
    regressors, accepted, rows, rhs = [], [], [], []
    margin = 1e-9
    for k in range(1, len(inputs)):
        forward = positions[k+1] - positions[k] > 2*noise_bound + margin
        compressed = inputs[k] + W - positions[k+1] > noise_bound + margin
        if not (forward and compressed):
            continue
        z = positions[k+1] - 2*positions[k] + positions[k-1]
        zeta = inputs[k] + W - 2*positions[k] + positions[k-1]
        normals = [[-zeta + 3*noise_bound, GRAVITY*H**2],
                   [ zeta + 3*noise_bound, -GRAVITY*H**2]]
        bounds = [4*noise_bound + allowance - z, 4*noise_bound + allowance + z]
        for normal, bound in zip(normals, bounds):
            polygon = clip_polygon(polygon, normal, bound)
            rows.append(normal)
            rhs.append(bound)
        regressors.append([zeta, -GRAVITY*H**2])
        accepted.append(k)
        sample_numbers.append(k+1)
        history.append(physical_bounds(polygon))
    if len(accepted) < 2:
        raise RuntimeError("Too few informative sliding-contact samples. Inspect the probe and noise level.")
    design = np.asarray(regressors)
    scales = np.linalg.norm(design, axis=0)
    normalized = design / np.maximum(scales, 1e-15)
    singular_values = np.linalg.svd(normalized, compute_uv=False)
    return {"polygon": polygon, "history": np.asarray(history),
            "sample_numbers": np.asarray(sample_numbers), "accepted": np.asarray(accepted),
            "rows": np.asarray(rows), "rhs": np.asarray(rhs),
            "singular_values": singular_values,
            "estimate": physical_parameters(polygon.mean(axis=0))}

print("Position-history feasible-set estimator ready.")


## Code cell 7 — Learn mass and friction; evaluate coverage separately


In [ ]:
fit = identify(measured_positions, commands)
estimated_mass, estimated_mu = fit["estimate"]
bounds = physical_bounds(fit["polygon"])

# Evaluation only, after identification has completed.
true_alpha = K_A * H**2 / (TRUE_MASS + K_A * H**2)
true_theta = np.array([true_alpha, TRUE_MU * (1-true_alpha)])
true_constraint_residual = fit["rows"] @ true_theta - fit["rhs"]
truth_consistent = bool(true_constraint_residual.max() <= 1e-12)
history = fit["history"]
true_physical = np.array([TRUE_MASS, TRUE_MU])
coverage = bool(np.all(history[:,:,0] <= true_physical + 1e-12)
                and np.all(history[:,:,1] >= true_physical - 1e-12))
assert truth_consistent and coverage, "Truth excluded: this matched-model check must pass."

print("Accepted sliding-contact transitions:", len(fit["accepted"]), "/", len(commands)-1)
print("Normalized regressor singular values:", fit["singular_values"])
print(f"Mass: true={TRUE_MASS*1000:.4f} g, estimate={estimated_mass*1000:.4f} g")
print(f"Mass bounds: [{bounds[0,0]*1000:.4f}, {bounds[0,1]*1000:.4f}] g")
print(f"Friction: true={TRUE_MU:.5f}, estimate={estimated_mu:.5f}")
print(f"Friction bounds: [{bounds[1,0]:.5f}, {bounds[1,1]:.5f}]")
print("Truth satisfies every accepted measurement constraint:", truth_consistent)
print("Truth inside all reported intervals:", coverage)
print("These bounds apply to the stated discrete model and noise assumptions.")


## Code cell 8 — Parameter intervals and information over time


In [ ]:
update_times = times[fit["sample_numbers"]]
plot_times = np.r_[update_times, times[-1]]
plot_history = np.concatenate([history, history[-1:]], axis=0)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for j, (scale, truth, label) in enumerate([(1000, TRUE_MASS, "Mass [g]"),
                                          (1, TRUE_MU, "Friction coefficient")]):
    axes[j].fill_between(plot_times, scale*plot_history[:,j,0], scale*plot_history[:,j,1],
                         step="post", alpha=.25, label="Feasible interval")
    axes[j].plot(plot_times, scale*plot_history[:,j,:].mean(axis=1),
                 drawstyle="steps-post", label="Interval midpoint")
    axes[j].axhline(scale*truth, color="k", ls="--", label="True value")
    axes[j].set_ylabel(label)
    axes[j].legend()
widths = plot_history[:,:,1] - plot_history[:,:,0]
axes[2].plot(plot_times, widths[:,0]/widths[0,0], drawstyle="steps-post", label="Mass")
axes[2].plot(plot_times, widths[:,1]/widths[0,1], drawstyle="steps-post", label="Friction")
axes[2].set_ylabel("Width / initial width")
axes[2].legend()
for ax in axes:
    ax.set_xlabel("Time [s]")
    ax.grid(alpha=.3)
plt.show()


## Code cell 9 — Prediction on a different command trajectory

This comparison does not update the estimator. All predictions receive the same new position commands and the same noisy initial-position measurement, with a known initial velocity of zero. The oracle knows the parameters but not the exact initial position. Scores use noiseless simulated positions for evaluation only.


In [ ]:
validation_knots = (2 * np.array([0, .10, .45, .65, 1.00, 1.20, 1.55, 1.80, 2.10])).tolist()
validation_offsets = [-.002, -.002, .014, .014, .040, .040, .065, -.002, -.002]
validation_times = np.arange(round(validation_knots[-1]/H)+1)*H
validation_rng = np.random.default_rng(SEED + 1000)
validation_start = 0.24
initial_reading = validation_start + validation_rng.uniform(-POSITION_NOISE, POSITION_NOISE)
validation_commands = smooth_schedule(validation_times[:-1], initial_reading,
                                       validation_knots, validation_offsets)
reference = rollout(validation_commands, TRUE_MASS, TRUE_MU, [validation_start, 0.0])
models = {"Before learning": (np.mean(MASS_PRIOR), np.mean(MU_PRIOR)),
          "After learning": (estimated_mass, estimated_mu), "Oracle": (TRUE_MASS, TRUE_MU)}
prediction_scores = {}
fig, axes = plt.subplots(1, 2, figsize=(12,4), constrained_layout=True)
axes[0].plot(validation_times, 1000*(reference["states"][:,0]-validation_start),
             "k--", label="Simulated plant")
for label, (mass, mu) in models.items():
    prediction = rollout(validation_commands, mass, mu, [initial_reading, 0.0])
    error = prediction["states"][:,0] - reference["states"][:,0]
    score = {"position_rmse_um": float(1e6*np.sqrt(np.mean(error**2))),
             "max_position_error_um": float(1e6*np.max(abs(error)))}
    prediction_scores[label] = score
    print(label, score)
    axes[0].plot(validation_times, 1000*(prediction["states"][:,0]-validation_start), label=label)
    axes[1].plot(validation_times, 1e6*error, label=label)
axes[0].set_ylabel("Object travel [mm]")
axes[1].set_ylabel("Prediction minus true position [um]")
for ax in axes:
    ax.set_xlabel("Time [s]")
    ax.grid(alpha=.3)
    ax.legend()
plt.show()


## Code cell 10 — Timestep sensitivity: an explicit limitation check

The estimator and plant above share the same implicit-Euler discretization. Small identification residuals do **not** establish accuracy relative to continuous physics. Here we compare $h$, $h/2$, and $h/4$ using the same continuous command schedule sampled at each rate. Differences include dynamics discretization and command-hold effects.

This is an empirical sensitivity diagnostic, not a certified model-error bound. Do not replace it by the tiny `MODEL_ALLOWANCE` used for the matched-model experiment. A later experiment should identify data generated by an independently refined plant and explicitly account for the discrepancy.


In [ ]:
refined_positions = []
for divisor in (1, 2, 4):
    local_h = H/divisor
    local_times = np.arange(round(KNOT_TIMES[-1]/local_h)+1)*local_h
    local_commands = smooth_schedule(local_times[:-1], observed_start, KNOT_TIMES, OFFSETS)
    local_run = rollout(local_commands, TRUE_MASS, TRUE_MU, INITIAL_STATE, h=local_h)
    refined_positions.append(local_run["states"][::divisor,0])
timestep_differences = []
for j, label in enumerate(["h versus h/2", "h/2 versus h/4"]):
    difference = float(np.max(abs(refined_positions[j]-refined_positions[j+1])))
    timestep_differences.append(difference)
    print(f"{label}: maximum position difference = {difference*1e6:.3f} um")
print("This diagnostic is separate from parameter-interval coverage in the matched discrete model.")


## Code cell 11 — Optional export into a new result folder


In [ ]:
SAVE_RESULTS = True  # Set True after reviewing the preceding cells, then run this cell.
if SAVE_RESULTS:
    project = Path.home() / "projects" / "adaptive_contact_linear_pusher"
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    output_dir = project / "results" / "dynamic_friction" / stamp
    output_dir.mkdir(parents=True, exist_ok=False)
    np.savez(output_dir / "identification.npz", times=times, commands=commands,
             measured_positions=measured_positions, true_states=experiment["states"],
             true_forces=experiment["forces"], estimated_parameters=fit["estimate"],
             parameter_polygon=fit["polygon"], interval_history=history,
             update_samples=fit["sample_numbers"], accepted_transitions=fit["accepted"])
    settings = {"model": "implicit Euler, massless spring, equal static/kinetic Coulomb friction",
                "K_A": K_A, "H": H, "W": W, "gravity": GRAVITY, "seed": SEED,
                "mass_prior_kg": MASS_PRIOR, "mu_prior": MU_PRIOR,
                "position_noise_bound_m": POSITION_NOISE, "model_allowance_m": MODEL_ALLOWANCE,
                "true_mass_kg": TRUE_MASS, "true_mu": TRUE_MU,
                "truth_consistent": truth_consistent, "interval_coverage": coverage,
                "prediction_scores": prediction_scores,
                "timestep_position_differences_m": timestep_differences}
    (output_dir / "settings.json").write_text(json.dumps(settings, indent=2))
    print("Saved:", output_dir)
else:
    print("Export disabled. Set SAVE_RESULTS=True when ready.")


## Interpreting this milestone

- The plant now has inertia, table friction, and changing contact conditions.
- Identification receives noisy object positions and known commands; it never receives simulated true velocity, force, mass, or friction.
- The parameter set keeps the correlation between mass-dependent response and friction; reported intervals are its physical-coordinate projections.
- Wide intervals or a small second regressor singular value indicate weak separation of the parameters. More samples alone are not a guarantee of useful excitation.
- The plots may show excellent matched-model predictions. Timestep sensitivity and real model mismatch remain separate questions.
- The probe is prescribed, the controller is not adaptive-SLS, and the polygon estimator is our baseline rather than the paper's zonotopic gain update. Next compare multiple masses/noise seeds, then account for refined-plant mismatch before optimizing the probe or integrating learning-aware feedback.

**Suggested first report:** physics checks, accepted sample count, estimated mass/friction and bounds, truth coverage, held-out prediction RMSE, and both timestep-sensitivity numbers. Keep the settings unchanged for the first run.
